# 03 - Modeling

### Austin Melendez & Sara Bruggman
###### Last Updated: 5/16/2026 1:54pm

This notebook loads the cleaned datasets from `01-preprocessing.ipynb`, and performs Welch's t-Tests, paired t-Tests, Mixed ANOVA, and ANCOVA.


In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
})

processed_dir <- file.path("data")
results_dir <- "results"
dir.create(results_dir, showWarnings = FALSE)

if (file.exists(file.path(processed_dir, "analysis_data.rds"))) {
  analysis_data <- readRDS(file.path(processed_dir, "analysis_data.rds"))
  survey_clean <- analysis_data$survey_clean
  pre_survey_clean <- analysis_data$pre_survey_clean
  post_survey_clean <- analysis_data$post_survey_clean
  matched_change <- analysis_data$matched_change
} else {
  survey_clean <- readr::read_csv(file.path(processed_dir, "survey_clean.csv"), show_col_types = FALSE)
  pre_survey_clean <- readr::read_csv(file.path(processed_dir, "pre_survey_clean.csv"), show_col_types = FALSE)
  post_survey_clean <- readr::read_csv(file.path(processed_dir, "post_survey_clean.csv"), show_col_types = FALSE)
  matched_change <- readr::read_csv(file.path(processed_dir, "matched_change.csv"), show_col_types = FALSE)
}

survey_clean <- survey_clean %>%
  mutate(
    Survey = factor(Survey, levels = c("Pre", "Post")),
    STEM = factor(STEM, levels = c("STEM", "Non-STEM")),
    Condition = factor(Condition, levels = c("Control", "Intervention")),
    `DID #` = factor(`DID #`)
  )

matched_change <- matched_change %>%
  mutate(
    STEM = factor(STEM, levels = c("STEM", "Non-STEM")),
    Condition = factor(Condition, levels = c("Control", "Intervention")),
    `DID #` = factor(`DID #`)
  )


## Helper Functions


In [ ]:
paired_d <- function(pre, post) {
  diff <- post - pre
  mean(diff, na.rm = TRUE) / sd(diff, na.rm = TRUE)
}

write_model_output <- function(..., path) {
  output <- capture.output(list(...))
  writeLines(output, con = path)
}


## Descriptive Summaries


In [ ]:
summary_by_group <- survey_clean %>%
  group_by(Survey, Condition, STEM) %>%
  summarise(
    n = n(),
    mean_J = mean(J_total, na.rm = TRUE),
    sd_J = sd(J_total, na.rm = TRUE),
    mean_T = mean(T_total, na.rm = TRUE),
    sd_T = sd(T_total, na.rm = TRUE),
    .groups = "drop"
  )

change_by_group <- matched_change %>%
  group_by(Condition, STEM) %>%
  summarise(
    n = n(),
    mean_J_change = mean(J_change, na.rm = TRUE),
    sd_J_change = sd(J_change, na.rm = TRUE),
    mean_T_change = mean(T_change, na.rm = TRUE),
    sd_T_change = sd(T_change, na.rm = TRUE),
    .groups = "drop"
  )

summary_by_group
change_by_group

readr::write_csv(summary_by_group, file.path(results_dir, "summary_by_group.csv"))
readr::write_csv(change_by_group, file.path(results_dir, "change_by_group.csv"))


## Baseline STEM vs Non-STEM Comparisons


In [ ]:
J_baseline_welch <- t.test(J_total ~ STEM, data = pre_survey_clean, var.equal = FALSE)
T_baseline_welch <- t.test(T_total ~ STEM, data = pre_survey_clean, var.equal = FALSE)

J_baseline_wilcox <- wilcox.test(J_total ~ STEM, data = pre_survey_clean)
T_baseline_wilcox <- wilcox.test(T_total ~ STEM, data = pre_survey_clean)

J_baseline_welch
T_baseline_welch
J_baseline_wilcox
T_baseline_wilcox


## Overall Paired Pre/Post Change


In [ ]:
J_paired_t <- t.test(pre_survey_clean$J_total, post_survey_clean$J_total, paired = TRUE)
T_paired_t <- t.test(pre_survey_clean$T_total, post_survey_clean$T_total, paired = TRUE)

J_paired_wilcox <- wilcox.test(pre_survey_clean$J_total, post_survey_clean$J_total, paired = TRUE)
T_paired_wilcox <- wilcox.test(pre_survey_clean$T_total, post_survey_clean$T_total, paired = TRUE)

J_paired_effect <- paired_d(pre_survey_clean$J_total, post_survey_clean$J_total)
T_paired_effect <- paired_d(pre_survey_clean$T_total, post_survey_clean$T_total)

J_paired_t
T_paired_t
J_paired_wilcox
T_paired_wilcox
J_paired_effect
T_paired_effect


## Intervention vs Control Change Comparisons


In [ ]:
J_condition_welch <- t.test(J_change ~ Condition, data = matched_change, var.equal = FALSE)
T_condition_welch <- t.test(T_change ~ Condition, data = matched_change, var.equal = FALSE)

J_condition_lm <- lm(J_change ~ Condition, data = matched_change)
T_condition_lm <- lm(T_change ~ Condition, data = matched_change)

J_condition_welch
T_condition_welch
summary(J_condition_lm)
summary(T_condition_lm)


## STEM Change Comparisons


In [ ]:
J_stem_change_welch <- t.test(J_change ~ STEM, data = matched_change, var.equal = FALSE)
T_stem_change_welch <- t.test(T_change ~ STEM, data = matched_change, var.equal = FALSE)

J_stem_change_lm <- lm(J_change ~ STEM, data = matched_change)
T_stem_change_lm <- lm(T_change ~ STEM, data = matched_change)

J_stem_change_welch
T_stem_change_welch
summary(J_stem_change_lm)
summary(T_stem_change_lm)


## Factorial Change-Score Models: Condition x STEM

These models test whether Intervention vs Control improvement differs by STEM status.


In [ ]:
J_change_interaction <- lm(J_change ~ Condition * STEM, data = matched_change)
T_change_interaction <- lm(T_change ~ Condition * STEM, data = matched_change)

summary(J_change_interaction)
anova(J_change_interaction)

summary(T_change_interaction)
anova(T_change_interaction)


## Mixed ANOVA: Survey x Condition x STEM

With two repeated time points, the time-by-between-subject effects are directly related to the change-score models. This repeated-measures ANOVA keeps the long-format structure and includes the subject-level error term.


In [ ]:
J_mixed_anova <- aov(J_total ~ Survey * Condition * STEM + Error(`DID #` / Survey), data = survey_clean)
T_mixed_anova <- aov(T_total ~ Survey * Condition * STEM + Error(`DID #` / Survey), data = survey_clean)

summary(J_mixed_anova)
summary(T_mixed_anova)


## ANCOVA: Post Score Adjusted for Baseline

ANCOVA estimates post-survey differences while adjusting for baseline score. This is a useful companion to change-score and mixed-ANOVA models.


In [ ]:
J_ancova <- lm(J_post ~ J_pre + Condition * STEM, data = matched_change)
T_ancova <- lm(T_post ~ T_pre + Condition * STEM, data = matched_change)

summary(J_ancova)
anova(J_ancova)

summary(T_ancova)
anova(T_ancova)


## Ceiling-Effect Models


In [ ]:
J_ceiling <- lm(J_change ~ J_pre, data = matched_change)
T_ceiling <- lm(T_change ~ T_pre, data = matched_change)

summary(J_ceiling)
summary(T_ceiling)

cor(matched_change$J_pre, matched_change$J_change, use = "complete.obs")
cor(matched_change$T_pre, matched_change$T_change, use = "complete.obs")


## Demographic Sensitivity Checks

These exploratory checks mirror comparisons from the original notebook. They are not the primary training-effect models.


In [ ]:
J_upper_lower <- t.test(J_total ~ `Upper v Lower`, data = pre_survey_clean, var.equal = FALSE)
T_upper_lower <- t.test(T_total ~ `Upper v Lower`, data = pre_survey_clean, var.equal = FALSE)

J_first_gen <- t.test(J_total ~ `First Generation Status`, data = pre_survey_clean, var.equal = FALSE)
T_first_gen <- t.test(T_total ~ `First Generation Status`, data = pre_survey_clean, var.equal = FALSE)

J_transfer <- t.test(J_total ~ `Transfer Student`, data = pre_survey_clean, var.equal = FALSE)
T_transfer <- t.test(T_total ~ `Transfer Student`, data = pre_survey_clean, var.equal = FALSE)

pre_gender_binary <- pre_survey_clean %>% filter(Gender %in% c("Man", "Woman"))
J_gender <- t.test(J_total ~ Gender, data = pre_gender_binary, var.equal = FALSE)
T_gender <- t.test(T_total ~ Gender, data = pre_gender_binary, var.equal = FALSE)

J_upper_lower
T_upper_lower
J_first_gen
T_first_gen
J_transfer
T_transfer
J_gender
T_gender


## Save Model Output


In [ ]:
model_objects <- list(
  baseline = list(J_welch = J_baseline_welch, T_welch = T_baseline_welch, J_wilcox = J_baseline_wilcox, T_wilcox = T_baseline_wilcox),
  paired = list(J_t = J_paired_t, T_t = T_paired_t, J_wilcox = J_paired_wilcox, T_wilcox = T_paired_wilcox, J_d = J_paired_effect, T_d = T_paired_effect),
  condition = list(J_welch = J_condition_welch, T_welch = T_condition_welch, J_lm = J_condition_lm, T_lm = T_condition_lm),
  stem_change = list(J_welch = J_stem_change_welch, T_welch = T_stem_change_welch, J_lm = J_stem_change_lm, T_lm = T_stem_change_lm),
  change_interaction = list(J = J_change_interaction, T = T_change_interaction),
  mixed_anova = list(J = J_mixed_anova, T = T_mixed_anova),
  ancova = list(J = J_ancova, T = T_ancova),
  ceiling = list(J = J_ceiling, T = T_ceiling)
)

saveRDS(model_objects, file.path(results_dir, "model_objects.rds"))

write_model_output(
  summary_by_group,
  change_by_group,
  J_baseline_welch,
  T_baseline_welch,
  J_paired_t,
  T_paired_t,
  summary(J_change_interaction),
  anova(J_change_interaction),
  summary(T_change_interaction),
  anova(T_change_interaction),
  summary(J_mixed_anova),
  summary(T_mixed_anova),
  summary(J_ancova),
  summary(T_ancova),
  path = file.path(results_dir, "model_output.txt")
)
